# Taste Test: GPT-5.5 in 60 Seconds

**Cold open for Segment 1.** No theory, no setup lecture — just five prompts so you can *feel* what GPT-5.5 does that older models couldn't.

Each demo targets a real GPT-5.5 strength:

1. **Schema-constrained JSON** — structured output you can parse without regex gymnastics.
2. **Multi-step reasoning** — an explicit chain-of-thought on a problem that trips up shallow models.
3. **Long-context compression** — paste ~300 words, get back a tight summary that preserves the load-bearing facts.
4. **Tool-use awareness** — the model designs a function-calling contract, showing it "thinks in tools."
5. **Style control** — a creative rewrite with a tone constraint.

These run against **`gpt-5.5`**, the course's default workhorse. We pin `reasoning_effort` explicitly on every call — the 5.x line moved its default twice (`medium` → `none` → `medium`), so relying on the default makes cost and latency unpredictable.

> **Note on `gpt-5.5-instant`.** The "instant" variant is primarily a ChatGPT product concept; its dedicated API name was unconfirmed at authoring time. We use the full `gpt-5.5` so this notebook runs reliably.

In [ ]:
from openai import OpenAI
import os, json

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Course default workhorse. (gpt-5.4-mini is the cheaper alternative if you're
# running many student exercises.)
MODEL = "gpt-5.5"


def ask(prompt, *, effort="none", verbosity="low", system=None):
    """Send one prompt and print the answer plus token usage.

    effort     -- reasoning_effort: none|low|medium|high|xhigh. Pinned on every
                  call so cost/latency are predictable (the 5.x default moved
                  twice across versions).
    verbosity  -- low|medium|high output-length steer.
    system     -- optional system/developer instruction.
    """
    input_items = []
    if system:
        input_items.append({"role": "system", "content": system})
    input_items.append({"role": "user", "content": prompt})

    r = client.responses.create(
        model=MODEL,
        input=input_items,
        reasoning={"effort": effort},
        text={"verbosity": verbosity},
    )

    print(r.output_text)
    print("-" * 60)

    u = r.usage
    line = f"tokens -> input: {u.input_tokens} | output: {u.output_tokens} | total: {u.total_tokens}"
    details = getattr(u, "output_tokens_details", None)
    reasoning_toks = getattr(details, "reasoning_tokens", None) if details else None
    if reasoning_toks:
        line += f"  (reasoning: {reasoning_toks})"
    print(line)
    print("=" * 60)
    return r

## Demo 1 — Schema-constrained JSON

**What we're showing:** GPT-5.5 can emit JSON that conforms to a schema *you* define, with no surrounding prose. **Why it matters:** in production you parse the response — you don't read it. Older models forced you to wrap calls in brittle regex or retry loops to strip markdown fences and apologies. Here we ask for a strict schema and then `json.loads()` the result directly. Watch that the output is valid JSON on the first try.

In [ ]:
# 1) Structured JSON output with schema validation
schema_prompt = """Extract structured data from this support ticket and return ONLY a JSON
object (no markdown, no prose) matching exactly this schema:

{
  "severity": "low" | "medium" | "high" | "critical",
  "category": string,
  "affected_service": string,
  "customer_sentiment": "calm" | "frustrated" | "angry",
  "action_items": [string, ...]
}

Ticket:
"This is the THIRD time checkout has gone down during a flash sale. We lost
an estimated $40k in the last hour. Your status page still says all systems
operational. Fix this NOW and tell me what you're doing to prevent it."
"""

r = ask(schema_prompt, effort="low", verbosity="low")

# Prove it's machine-parseable: load it and pull a field out.
parsed = json.loads(r.output_text)
print("\nParsed OK. Severity =", parsed["severity"],
      "| sentiment =", parsed["customer_sentiment"])
print("Action items:")
for item in parsed["action_items"]:
    print("  -", item)

## Demo 2 — Multi-step reasoning with explicit chain-of-thought

**What we're showing:** a word problem with interacting constraints that shallow, pattern-matching answers get wrong. We bump `reasoning_effort` to `medium` and ask the model to show its work step by step. **Why it matters:** for anything with arithmetic or multi-hop logic, visible reasoning is both more accurate *and* auditable — you can see exactly where a conclusion came from. Note the `reasoning` token count in the usage line: that's the "thinking" you're paying for, and why you don't turn effort up for trivial tasks.

In [ ]:
# 2) Multi-step reasoning with explicit chain-of-thought
reasoning_prompt = """Three servers (A, B, C) share a load balancer.

- A handles twice as many requests per second as B.
- C handles 50 more requests per second than A.
- Together they handle 1,050 requests per second.
- Each server can safely sustain 500 req/s before it needs to scale out.

Work through this step by step. Then state: the req/s for each server, and
which servers (if any) are over their safe threshold.
"""

ask(reasoning_prompt, effort="medium", verbosity="medium")

## Demo 3 — Long-context summarization

**What we're showing:** paste a ~300-word incident writeup and ask for a 3-bullet executive summary that keeps the numbers and the root cause. **Why it matters:** GPT-5.5 has a ~1M-token context window, but the skill on display here is *faithful compression* — preserving the load-bearing facts (dates, dollar figures, the actual cause) while dropping the narrative padding. Watch the input-token count jump versus the earlier demos: that's the cost of the longer prompt, and the output stays tight because we asked for it.

In [ ]:
# 3) Long-context summarization -- compress a ~300-word blurb
blurb = """On March 14th at 02:17 UTC, our payments service began returning HTTP 503s
for roughly 38% of checkout attempts. The on-call engineer was paged at 02:24 and
acknowledged at 02:29. Initial dashboards showed elevated latency on the primary
Postgres instance but no obvious spike in traffic; load was actually 12% below the
trailing 7-day average for that hour. At 02:41 the team noticed that the connection
pool on the payments service had saturated at its max of 200 connections, with most
connections stuck in an 'idle in transaction' state. Root cause analysis later traced
this to a deploy at 01:55 that introduced a code path which opened a transaction to
write an audit log, called an external fraud-scoring API inside that transaction, and
only committed after the API returned. When the fraud API's p99 latency rose to 9
seconds during a provider-side incident, each checkout held a database transaction
open for the full duration, exhausting the pool. Mitigation came in two steps: at 02:48
the team raised the pool ceiling to 400 (buying headroom), and at 03:10 they shipped a
hotfix moving the fraud API call outside the transaction and making the audit write
asynchronous. Error rates returned to baseline by 03:19. Total customer-facing impact
was 62 minutes; an estimated 4,100 checkout attempts failed, of which about 2,700 were
retried successfully by customers within the hour. Estimated lost revenue was $58,000.
Follow-up actions: add a lint rule forbidding network calls inside DB transactions, add
a pool-saturation alert at 80% utilization, and add a synthetic checkout probe that runs
every 30 seconds independent of real traffic.
"""

summary_prompt = (
    "Summarize this incident report in exactly 3 bullets for an executive audience. "
    "Keep the root cause, the customer impact (time + dollars), and the top prevention "
    "action. Drop the play-by-play timeline.\n\n" + blurb
)

ask(summary_prompt, effort="low", verbosity="low")

## Demo 4 — Tool-use awareness

**What we're showing:** without actually wiring up any tools, we ask the model to *design* a function-calling contract — the JSON schema for a tool plus when it should and shouldn't be called. **Why it matters:** GPT-5.5 is trained to "think in tools," which is the foundation of every agentic workflow in the rest of this course. Seeing it reason about parameter types, required fields, and guard conditions tells you it will play well as the planner in a real tool-calling loop. (Later notebooks turn this into live function calling.)

In [ ]:
# 4) Tool-use awareness -- have the model design a function-calling contract
tool_prompt = """You are designing a function/tool that a support assistant can call.
The tool reschedules a customer's delivery.

Produce:
1. A JSON Schema for the tool's parameters (OpenAI function-calling style), with
   correct types and a `required` list.
2. Three concrete example user messages where the assistant SHOULD call this tool,
   and the exact arguments it would pass.
3. Two example user messages where it should NOT call the tool, and why (e.g.
   missing info that must be asked for first).

Be precise about parameter types and validation."""

ask(tool_prompt, effort="medium", verbosity="medium")

## Demo 5 — Style and tone control

**What we're showing:** the same underlying message ("our service had an outage") rewritten under a tight tone constraint via a system instruction. **Why it matters:** controllable voice is what lets you put a model in front of customers. We hold the facts fixed and steer only the register — calm, honest, no corporate weasel-words — and bump verbosity up so the model has room to land the tone. This is the creative end of the spectrum, but with guardrails, which is exactly how you'd use it in production copy.

In [ ]:
# 5) Style + tone control via a system instruction
style_system = (
    "You write public status-page updates. Voice: calm, plain-spoken, honest. "
    "Never use corporate weasel-words ('some users may have experienced'), never "
    "minimize, never over-apologize. Lead with what happened, then impact, then "
    "what we're doing. Keep it human."
)

style_prompt = (
    "Write a status-page update for this: our checkout service was fully down for "
    "62 minutes overnight due to a database connection-pool exhaustion bug we "
    "introduced in a deploy. ~4,100 checkouts failed. It's fixed and we've added "
    "alerting to catch it earlier next time."
)

ask(style_prompt, effort="none", verbosity="high", system=style_system)

## That's the taste test

Five demos, each hitting something a frontier model does well and a cheaper/older one struggles with:

- **Schema-constrained JSON** you can `json.loads()` without cleanup.
- **Multi-step reasoning** with visible, auditable work — and a token bill you could see.
- **Faithful long-context compression** that kept the numbers and root cause.
- **Tool-use awareness** — the model designed a function-calling contract on its own.
- **Tone control** — same facts, steered voice, production-ready copy.

You also saw the two knobs we'll lean on all course long, pinned explicitly on every call: **`reasoning_effort`** (none → xhigh) and **`verbosity`** (low → high). Next segment we slow down and explain *why* the model behaves this way — and how the Responses API exposes all of it.